<a href="https://colab.research.google.com/github/AniShriyak/Bio-Inspired-Lab/blob/main/BIS_CuckooSearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import random
import numpy as np


class CuckooSearchVRP:

    def __init__(
        self,
        depot,
        customers,
        demands,
        vehicle_capacity,
        n_nests=25,
        pa=0.25,
        max_iter=200,
    ):
        """Parameters:

        - depot: (x, y) coordinates of the central depot
        - customers: List of (x, y) tuples for customer locations
        - demands: List of demands for each customer
        - vehicle_capacity: Maximum load a single vehicle can carry
        - n_nests: Population size of nests (cuckoos)
        - pa: Discovery probability of host abandoning alien eggs
        - max_iter: Maximum number of iterations
        """
        self.depot = depot
        self.customers = customers
        self.demands = demands
        self.capacity = vehicle_capacity
        self.n_customers = len(customers)
        self.n_nests = n_nests
        self.pa = pa
        self.max_iter = max_iter

        # Precompute distance matrix (Depot = Index 0)
        self.nodes = [depot] + customers
        self.dist_matrix = self._compute_distance_matrix()

    def _compute_distance_matrix(self):
        n = len(self.nodes)
        matrix = np.zeros((n, n))
        for i in range(n):
            for j in range(n):
                p1, p2 = self.nodes[i], self.nodes[j]
                matrix[i][j] = math.hypot(p1[0] - p2[0], p1[1] - p2[1])
        return matrix

    def decode_permutation_to_routes(self, perm):
        """Split a giant tour (permutation of customer indices) into valid

        capacity-constrained vehicle routes.
        """
        routes = []
        current_route = []
        current_load = 0

        for cust_idx in perm:
            demand = self.demands[cust_idx]
            if current_load + demand <= self.capacity:
                current_route.append(cust_idx + 1)  # +1 due to depot at 0
                current_load += demand
            else:
                routes.append(current_route)
                current_route = [cust_idx + 1]
                current_load = demand

        if current_route:
            routes.append(current_route)
        return routes

    def calculate_fitness(self, perm):
        """Calculate total route distance across all vehicles."""
        routes = self.decode_permutation_to_routes(perm)
        total_dist = 0.0

        for route in routes:
            # Distance from depot to first customer
            total_dist += self.dist_matrix[0][route[0]]

            # Distance between consecutive customers
            for i in range(len(route) - 1):
                total_dist += self.dist_matrix[route[i]][route[i + 1]]

            # Distance from last customer back to depot
            total_dist += self.dist_matrix[route[-1]][0]

        return total_dist

    def _levy_flight_step(self, perm):
        """Discrete Lévy Flight: Performs heavy-tailed structural perturbations

        (2-Opt, Swaps, or Segment Shifts) based on step length.
        """
        # Mantegna algorithm for Levy distribution step length
        beta = 1.5
        sigma_u = (
            math.gamma(1 + beta)
            * math.sin(math.pi * beta / 2)
            / (
                math.gamma((1 + beta) / 2)
                * beta
                * (2 ** ((beta - 1) / 2))
            )
        ) ** (1 / beta)
        u = random.gauss(0, sigma_u)
        v = random.gauss(0, 1)
        step = u / (abs(v) ** (1 / beta))

        # Perform mutations based on the magnitude of the step
        new_perm = list(perm)
        k = max(1, int(abs(step) % (self.n_customers // 2)))

        for _ in range(k):
            choice = random.random()
            i, j = sorted(random.sample(range(self.n_customers), 2))
            if choice < 0.5:
                # 2-Opt (Invert segment)
                new_perm[i : j + 1] = reversed(new_perm[i : j + 1])
            else:
                # Swap two random customer positions
                new_perm[i], new_perm[j] = new_perm[j], new_perm[i]

        return new_perm

    def solve(self):
        # Step 1: Initialize population (Giant Tours)
        nests = [
            random.sample(range(self.n_customers), self.n_customers)
            for _ in range(self.n_nests)
        ]
        fitness = [self.calculate_fitness(n) for n in nests]

        best_idx = np.argmin(fitness)
        best_nest = list(nests[best_idx])
        best_fitness = fitness[best_idx]

        # Optimization Loop
        for iteration in range(self.max_iter):
            # Phase 1: Generate new solutions via Discrete Lévy Flights
            for i in range(self.n_nests):
                new_nest = self._levy_flight_step(nests[i])
                new_fit = self.calculate_fitness(new_nest)

                # Greedy selection against a random nest j
                j = random.randint(0, self.n_nests - 1)
                if new_fit < fitness[j]:
                    nests[j] = new_nest
                    fitness[j] = new_fit

            # Phase 2: Abandon fraction pa of worst nests
            worst_count = int(self.pa * self.n_nests)
            sorted_indices = np.argsort(fitness)

            for idx in sorted_indices[-worst_count:]:
                # Replace abandoned nests with newly randomized perturbation
                i, j = sorted(random.sample(range(self.n_customers), 2))
                new_nest = list(nests[idx])
                new_nest[i:j] = reversed(new_nest[i:j])
                nests[idx] = new_nest
                fitness[idx] = self.calculate_fitness(new_nest)

            # Update overall best nest
            current_best_idx = np.argmin(fitness)
            if fitness[current_best_idx] < best_fitness:
                best_fitness = fitness[current_best_idx]
                best_nest = list(nests[current_best_idx])

            if (iteration + 1) % 50 == 0 or iteration == 0:
                print(
                    f"Iteration {iteration + 1}/{self.max_iter} | Best Distance: {best_fitness:.2f}"
                )

        best_routes = self.decode_permutation_to_routes(best_nest)
        return best_nest, best_routes, best_fitness


# ---------------------------------------------------------
# Example Run
# ---------------------------------------------------------
if __name__ == "__main__":
    random.seed(42)
    np.random.seed(42)

    # Depot and 10 Customers with coordinates
    depot_loc = (50, 50)
    customer_locs = [
        (20, 80),
        (30, 50),
        (80, 70),
        (70, 20),
        (90, 30),
        (20, 30),
        (50, 80),
        (40, 10),
        (10, 60),
        (60, 40),
    ]
    customer_demands = [15, 20, 10, 30, 25, 10, 15, 20, 10, 15]
    max_vehicle_capacity = 50

    solver = CuckooSearchVRP(
        depot=depot_loc,
        customers=customer_locs,
        demands=customer_demands,
        vehicle_capacity=max_vehicle_capacity,
        n_nests=30,
        pa=0.25,
        max_iter=200,
    )

    best_perm, routes, total_dist = solver.solve()

    print("\n--- Optimization Result ---")
    print(f"Total Travel Distance: {total_dist:.2f}")
    print(f"Number of Vehicles Required: {len(routes)}")
    for i, route in enumerate(routes, 1):
        load = sum(customer_demands[c - 1] for c in route)
        print(f"  Vehicle {i} Route: Depot -> {' -> '.join(map(str, route))} -> Depot (Load: {load}/{max_vehicle_capacity})")

Iteration 1/200 | Best Distance: 488.53
Iteration 50/200 | Best Distance: 422.00
Iteration 100/200 | Best Distance: 422.00
Iteration 150/200 | Best Distance: 422.00
Iteration 200/200 | Best Distance: 422.00

--- Optimization Result ---
Total Travel Distance: 422.00
Number of Vehicles Required: 4
  Vehicle 1 Route: Depot -> 4 -> 8 -> Depot (Load: 50/50)
  Vehicle 2 Route: Depot -> 10 -> 5 -> 3 -> Depot (Load: 50/50)
  Vehicle 3 Route: Depot -> 6 -> 9 -> 1 -> 7 -> Depot (Load: 50/50)
  Vehicle 4 Route: Depot -> 2 -> Depot (Load: 20/50)
